# 12 — Textbook ensembles (Géron Ch.7), done right → cross-family stacking
The course deck `glava7` is Hands-On ML **Chapter 7**: voting → bagging/pasting + OOB → random patches/subspaces → Random Forests → Extra-Trees → AdaBoost → GBRT → **stacking**. This notebook walks **every** method on the real sleep-stage task, **honestly on the shared folds** (macro-F1, no leakage), then does *the same but better*: the textbook's climax — **stacking a meta-learner over diverse base models** — applied **across model families**, the project's hard-won lesson that single-family tree ensembles hit a ~0.83 wall.

Reused (already trained on these exact folds → valid leak-free stacking inputs): CatBoost, SVM-RBF bag, QDA, GMM, RF/ET/GB. Trained live: the textbook learners + the blender.

In [ ]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [ ]:
def log_result(step, model, feature_set, oof_macro, pcf, notes=""):
    """Write one row to results_log.csv. Idempotent per (step, model): re-running
    a notebook replaces its own row rather than duplicating it."""
    import csv
    path = HERE / "results_log.csv"
    header = ["step", "model", "feature_set", "oof_macro_f1",
              "f1_Wake", "f1_Light", "f1_Deep", "f1_REM", "notes"]
    rows = []
    if path.exists():
        with open(path, newline="") as fh:
            data = list(csv.reader(fh))
        if data and data[0] == header:
            rows = [r for r in data[1:] if not (len(r) >= 2 and r[0] == step and r[1] == model)]
    row = [step, model, feature_set, round(float(oof_macro), 5),
           pcf["Wake"], pcf["Light"], pcf["Deep"], pcf["REM"], notes]
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header); w.writerows(rows); w.writerow(row)
    print("logged:", step, model, "OOF macro-F1 =", round(float(oof_macro), 5))


In [ ]:
from sklearn.model_selection import cross_val_predict, RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, VotingClassifier,
                              BaggingClassifier, StackingClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from catboost import CatBoostClassifier

cols = json.load(open(ART / "feature_cols.json"))["columns"]
EOG_I = cols.index(EOG)
Xtr = pd.DataFrame(np.load(ART / "features_v1_train.npy"), columns=cols)
Xte = pd.DataFrame(np.load(ART / "features_v1_test.npy"), columns=cols)
Xtr_np, Xte_np = Xtr.values, Xte.values
y = np.load(ART / "y_train.npy")
test_ids = np.load(ART / "test_ids.npy")
folds = load_folds()
gate = json.load(open(ART / "repeated_cv_gate.json"))
NOISE = gate["std"]                                   # accept/reject noise floor (~0.0017)

# Members already trained on the SHARED folds (row-aligned) -> reused as stacking inputs, never retrained.
# Guarded load so a partial pipeline degrades gracefully instead of crashing; the deployed-stack members
# are asserted present (the rest are illustrative).
SAVED = ["catboost_tuned", "svm", "rf", "et", "gb", "qda", "gmm2", "nystroem", "mlp"]
REQUIRED = ["catboost_tuned", "svm", "qda", "gmm2"]
oof, tst, missing = {}, {}, []
for n in SAVED:
    if (ART / f"{n}_oof.npy").exists() and (ART / f"{n}_test.npy").exists():
        oof[n] = np.load(ART / f"{n}_oof.npy"); tst[n] = np.load(ART / f"{n}_test.npy")
    else:
        missing.append(n)
if missing:
    print("WARNING: missing member artifacts (run steps 04-11 first):", missing)
assert all(n in oof for n in REQUIRED), f"deployed-stack members missing: {[n for n in REQUIRED if n not in oof]}"
anchor = macro_f1(y, oof["catboost_tuned"].argmax(1))
print("reused members (OOF macro-F1 on shared folds):")
for n in SAVED:
    print(f"  {n:16s} {macro_f1(y, oof[n].argmax(1)):.5f}")
print(f"\nanchor (CatBoost) {anchor:.5f} | noise floor {NOISE:.5f} | sub06 Cat+SVM 0.83269")


def aligned(model, X):
    """predict_proba with columns ordered to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X); cls = list(np.asarray(model.classes_))
    return p[:, [cls.index(c) for c in CLASSES]]


def oof_predict(make_model, impute=True, save_as=None):
    """OOF + averaged-test probabilities for a fresh sklearn learner on the SHARED folds.
    impute=True  -> fill the EOG NaN with the TRAIN-FOLD median (sklearn learners reject NaN);
    impute=False -> pass NaN through (HistGradientBoosting handles it natively)."""
    oo = np.zeros((len(y), len(CLASSES))); te = np.zeros((len(Xte), len(CLASSES)))
    for tr, va in folds:
        Xt, Xv, Xe = Xtr_np[tr].copy(), Xtr_np[va].copy(), Xte_np.copy()
        if impute:
            med = np.nanmedian(Xt[:, EOG_I])
            for M in (Xt, Xv, Xe):
                c = M[:, EOG_I]; c[np.isnan(c)] = med
        m = make_model(); m.fit(Xt, y[tr])
        oo[va] = aligned(m, Xv); te += aligned(m, Xe) / len(folds)
    if save_as:
        np.save(ART / f"{save_as}_oof.npy", oo); np.save(ART / f"{save_as}_test.npy", te)
    return oo, te


class GMMClassifier:
    """Generative Bayes (matches 11_multifamily): one full-covariance GaussianMixture per class,
    posterior proportional to prior * density. Used as a decorrelated stacking input."""
    def __init__(self, n_components=2, reg_covar=1e-3, seed=SEED):
        self.k, self.reg, self.seed = n_components, reg_covar, seed
    def fit(self, X, yv):
        self.classes_ = np.unique(yv); self.gmms_, self.logpri_ = [], []
        for c in self.classes_:
            g = GaussianMixture(self.k, covariance_type="full", reg_covar=self.reg,
                                random_state=self.seed, max_iter=200).fit(X[yv == c])
            self.gmms_.append(g); self.logpri_.append(np.log((yv == c).mean()))
        return self
    def predict_proba(self, X):
        logp = np.column_stack([g.score_samples(X) + lp for g, lp in zip(self.gmms_, self.logpri_)])
        logp -= logp.max(1, keepdims=True); p = np.exp(logp)
        return p / p.sum(1, keepdims=True)


## (1) Voting — hard vs soft (slides 3–7)
Soft voting (averaging probabilities) usually beats hard voting — **but only when the members' probabilities are calibrated and their errors independent**. Honest OOF makes the caveat concrete (a miscalibrated GaussianNB here pushes soft *below* hard), and the punchline lands: two *strong, decorrelated* members beat five mediocre ones — why the step-06 soft-voting `sub02` (0.81952) hurt.

In [ ]:
# Slides 4/6: diverse base learners -> hard vs soft voting. Each learner is a leak-safe pipeline
# (median-impute the EOG NaN + StandardScaler fit INSIDE each train fold). Scored on the shared-fold
# OOF with the competition metric -> an honest read of the slide, not a single lucky split.
pipe = lambda est: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), est)
base_learners = [
    ("logreg", pipe(LogisticRegression(max_iter=2000, C=1.0))),
    ("rf",     pipe(RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))),
    ("extra",  pipe(ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))),
    ("knn",    pipe(KNeighborsClassifier(n_neighbors=25))),
    ("gnb",    pipe(GaussianNB())),
]
indiv = {}
print("individual base learners (OOF macro-F1):")
for name, est in base_learners:
    indiv[name] = macro_f1(y, cross_val_predict(est, Xtr_np, y, cv=folds, method="predict"))
    print(f"  {name:8s} {indiv[name]:.5f}")
best_member = max(indiv, key=indiv.get)

hard = VotingClassifier(base_learners, voting="hard")
soft = VotingClassifier(base_learners, voting="soft")
mh = macro_f1(y, cross_val_predict(hard, Xtr_np, y, cv=folds, method="predict"))
ms = macro_f1(y, cross_val_predict(soft, Xtr_np, y, cv=folds, method="predict"))
print(f"\nHARD voting OOF macro-F1 = {mh:.5f}")
print(f"SOFT voting OOF macro-F1 = {ms:.5f}")
if ms >= mh:
    print("  -> soft >= hard (slide 6): averaging calibrated probabilities up-weights confident, correct votes.")
else:
    print("  -> soft < hard HERE: soft voting averages PROBABILITIES, so one MISCALIBRATED member poisons the mean.")
    print(f"     GaussianNB (OOF {indiv['gnb']:.3f}) emits over-confident near-0/1 probs and drags the soft average")
    print("     below the hard majority vote -- the slide-6 caveat ('only if the votes are meaningful'), live.")
print(f"Both voting variants sit BELOW the best single member ({best_member} {indiv[best_member]:.5f}): weak/correlated learners dilute a vote.")

# The fix the slide hints at: vote a FEW STRONG, DECORRELATED members instead of many mediocre ones --
# a soft vote of just CatBoost + SVM beats the 5-learner vote. Count loses to diversity-of-strength;
# this is exactly why the step-06 soft-voting sub02 (0.81952) over weak, correlated trees HURT.
cf_vote = (oof["catboost_tuned"] + oof["svm"]) / 2
print(f"\nsoft-vote CatBoost+SVM (2 strong, decorrelated) = {macro_f1(y, cf_vote.argmax(1)):.5f}  >> the 5-learner vote.")

## (2) Bagging vs pasting + free OOB (slides 8–12)
Bagging (bootstrap) trades a little bias for lower variance; pasting samples without replacement. Each tree leaves out ~37% of rows → an **out-of-bag** estimate that needs no validation split. We score OOB on the *competition metric* via `oob_decision_function_` and confirm it matches the CV.

In [ ]:
# Slides 8-12: bagging (bootstrap=True) vs pasting (bootstrap=False), plus the FREE out-of-bag score.
# Base = unpruned decision trees (high variance -> bagging shines). For pasting we sample 63% of rows
# without replacement so the trees actually differ (full-sample pasting would clone one tree).
N_TREES = 400

def make_bag(bootstrap, max_samples):
    return BaggingClassifier(DecisionTreeClassifier(random_state=SEED), n_estimators=N_TREES,
                             bootstrap=bootstrap, max_samples=max_samples, n_jobs=-1, random_state=SEED)

p_bag   = cross_val_predict(make_pipeline(SimpleImputer(strategy="median"), make_bag(True, 1.0)),  Xtr_np, y, cv=folds)
p_paste = cross_val_predict(make_pipeline(SimpleImputer(strategy="median"), make_bag(False, 0.63)), Xtr_np, y, cv=folds)
print(f"bagging (bootstrap=True)  OOF macro-F1 = {macro_f1(y, p_bag):.5f}")
print(f"pasting (bootstrap=False) OOF macro-F1 = {macro_f1(y, p_paste):.5f}   (slide 9: bagging usually >=)")

# OOB (slide 12): each tree leaves out ~37% of rows -> evaluate WITHOUT a validation split. Default
# oob_score_ is accuracy; we recompute the COMPETITION metric from oob_decision_function_.
Xi = Xtr_np.copy(); Xi[np.isnan(Xi[:, EOG_I]), EOG_I] = np.nanmedian(Xtr_np[:, EOG_I])
bag_oob = BaggingClassifier(DecisionTreeClassifier(random_state=SEED), n_estimators=N_TREES,
                            bootstrap=True, oob_score=True, n_jobs=-1, random_state=SEED).fit(Xi, y)
print(f"\nOOB accuracy = {bag_oob.oob_score_:.4f} | OOB macro-F1 = {macro_f1(y, bag_oob.oob_decision_function_.argmax(1)):.5f}")
print(f"   matches the 5-fold OOF bagging {macro_f1(y, p_bag):.5f} -> the free OOB estimate is trustworthy.")

## (3) Random patches & subspaces (slide 13)
Sampling features (`max_features`, `bootstrap_features`) adds diversity — decisive in high dimensions. On 28 curated features we **measure** it: feature subsampling does help (it is exactly what turns plain bagging into a Random Forest), while extra row-sampling on top adds little.

In [ ]:
# Slide 13: random patches (sample rows AND features) and random subspaces (all rows, sample
# features). We MEASURE the effect on these 28 features instead of assuming it.
configs = {
    "all features (plain bagging)":   dict(max_features=1.0, bootstrap_features=False, max_samples=1.0,  bootstrap=True),
    "random subspaces (feat=0.5)":    dict(max_features=0.5, bootstrap_features=False, max_samples=1.0,  bootstrap=False),
    "random patches (rows+feat=0.5)": dict(max_features=0.5, bootstrap_features=True,  max_samples=0.63, bootstrap=True),
}
patch_sc = {}
for name, kw in configs.items():
    est = make_pipeline(SimpleImputer(strategy="median"),
                        BaggingClassifier(DecisionTreeClassifier(random_state=SEED), n_estimators=N_TREES,
                                          n_jobs=-1, random_state=SEED, **kw))
    patch_sc[name] = macro_f1(y, cross_val_predict(est, Xtr_np, y, cv=folds))
    print(f"  {name:32s} OOF macro-F1 = {patch_sc[name]:.5f}")
plain = patch_sc["all features (plain bagging)"]; subsp = patch_sc["random subspaces (feat=0.5)"]
rf_macro = macro_f1(y, oof["rf"].argmax(1))
print(f"-> feature subsampling moves macro-F1 by {subsp - plain:+.5f} vs plain bagging "
      f"({'helps' if subsp > plain else 'no gain'}).")
print(f"   Decorrelating trees by sampling features IS what a Random Forest does: subspaces {subsp:.4f} ~ RF {rf_macro:.4f}.")
print("   Row-sampling on top (patches) adds little here; the largest subspace wins need genuine high-D (slide 13: images).")

## (4) Random Forest, Extra-Trees & feature importance (slides 14–19)
Reuse the RF/ET OOF from `04_models`; both sit well below CatBoost/SVM (single-family trees aren't enough). A forest also gives a free importance ranking — confirming the EOG burst channel #1.

In [ ]:
# Slides 14-19: Random Forest vs Extra-Trees + the FREE feature-importance read-out. RF/ET OOF were
# already computed on these folds in 04_models -> reused (no retrain).
print(f"RandomForest OOF macro-F1 = {macro_f1(y, oof['rf'].argmax(1)):.5f}")
print(f"Extra-Trees  OOF macro-F1 = {macro_f1(y, oof['et'].argmax(1)):.5f}")
print("(both far below CatBoost/SVM -> single-family tree bagging is not enough on this task.)")

rf_full = RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
imp_X = Xtr_np.copy(); imp_X[np.isnan(imp_X[:, EOG_I]), EOG_I] = np.nanmedian(Xtr_np[:, EOG_I])
rf_full.fit(imp_X, y)
order = np.argsort(rf_full.feature_importances_)[::-1]
print("\ntop-10 features by RF impurity-decrease importance:")
for i in order[:10]:
    print(f"  {cols[i]:24s} {rf_full.feature_importances_[i]:.4f}")
print(f"\n#1 feature = {cols[order[0]]} -> the EOG burst channel dominates (matches the EDA).")

## (5) Boosting — AdaBoost & GBRT (slides 20–26)
Sequential error-correction. AdaBoost (SAMME) re-weights mistakes; stochastic GBRT uses shrinkage + subsampling with **early stopping via `staged_predict`** (the metric-aware tree count per fold). `HistGradientBoosting` is the modern NaN-native GBRT; CatBoost remains the boosting champion here.

In [ ]:
# Slides 20-26: boosting trains learners SEQUENTIALLY, each fixing the predecessor.
# (a) AdaBoost (SAMME): re-weights misclassified samples; shallow-tree base.
ada_oof, ada_test = oof_predict(
    lambda: AdaBoostClassifier(DecisionTreeClassifier(max_depth=2, random_state=SEED),
                               n_estimators=300, learning_rate=0.5, random_state=SEED),
    impute=True, save_as="adaboost")
print(f"AdaBoost(SAMME, depth-2 x300)            OOF macro-F1 = {macro_f1(y, ada_oof.argmax(1)):.5f}")

# (b) Stochastic GBRT (shrinkage lr=0.05 + subsample=0.8) with EARLY STOPPING via staged_predict
# (slide 26): pick the tree count that maximises macro-F1 on each fold's held-out set.
def gbrt_oof_earlystop(n_max=400):
    oo = np.zeros((len(y), len(CLASSES))); te = np.zeros((len(Xte), len(CLASSES))); best_ns = []
    for tr, va in folds:
        Xt, Xv, Xe = Xtr_np[tr].copy(), Xtr_np[va].copy(), Xte_np.copy()
        med = np.nanmedian(Xt[:, EOG_I])
        for M in (Xt, Xv, Xe):
            c = M[:, EOG_I]; c[np.isnan(c)] = med
        g = GradientBoostingClassifier(n_estimators=n_max, learning_rate=0.05, max_depth=3,
                                       subsample=0.8, random_state=SEED).fit(Xt, y[tr])
        stage = [macro_f1(y[va], p) for p in g.staged_predict(Xv)]
        best_n = int(np.argmax(stage)) + 1; best_ns.append(best_n)
        idx = [list(g.classes_).index(c) for c in CLASSES]
        oo[va] = list(g.staged_predict_proba(Xv))[best_n - 1][:, idx]
        te += list(g.staged_predict_proba(Xe))[best_n - 1][:, idx] / len(folds)
    return oo, te, best_ns

gb_oof2, gb_test2, best_ns = gbrt_oof_earlystop()
print(f"GBRT (lr=.05, subsample=.8, early-stop)  OOF macro-F1 = {macro_f1(y, gb_oof2.argmax(1)):.5f}")
print(f"  per-fold optimal tree count (staged_predict): {best_ns}")

# (c) HistGradientBoosting -- modern, NaN-native, much faster GBRT (no imputation).
hist_oof, hist_test = oof_predict(
    lambda: HistGradientBoostingClassifier(max_iter=600, learning_rate=0.05, max_leaf_nodes=31,
                                           l2_regularization=1.0, random_state=SEED),
    impute=False, save_as="histgb")
print(f"HistGradientBoosting (NaN-native)        OOF macro-F1 = {macro_f1(y, hist_oof.argmax(1)):.5f}")
print(f"CatBoost (production GBDT, step-04)       OOF macro-F1 = {anchor:.5f}  <- the boosting champion here")
oof["adaboost"], tst["adaboost"] = ada_oof, ada_test
oof["histgb"], tst["histgb"] = hist_oof, hist_test

## (6) Stacking — the climax (slides 27–28): *do the same, but better*
Replace the fixed vote with a **trained blender**. First the faithful `StackingClassifier` API on tree learners (still stuck at the wall), then **diagnostic** layered stacks that show *where* the lift comes from — it appears only when we cross **families**. The deployed member set (CatBoost + SVM + *decorrelated* QDA/GMM) is **fixed a-priori from the step-11 eligibility analysis** (strong + most-decorrelated), **not** picked by ranking the layer OOFs — that ranking is illustrative only; the honest validation is the paired CV in §7. The blender is fit strictly out-of-fold (`cross_val_predict`); meta `C` fixed a-priori (in-sample C-stability only sanity-checked). Adding the set-aside MLP (shown, not shipped) is the one remaining lever.

In [ ]:
# Slides 27-28: STACKING -- instead of a fixed vote, TRAIN a meta-learner (blender) to combine base
# predictions. Honest recipe = base models predict on HELD-OUT folds, the blender trains on those clean
# predictions. Every member here was produced on the SAME shared folds, so the saved OOF matrices ARE
# leak-free meta-features; the blender (multinomial LogReg) is fit out-of-fold via cross_val_predict.
# Meta C fixed a-priori = 1.0 (no OOF tuning); its in-sample C-stability is sanity-checked below.
def stack_oof_test(names, C=1.0):
    Xo = np.hstack([oof[n] for n in names]); Xt = np.hstack([tst[n] for n in names])
    meta = LogisticRegression(max_iter=4000, C=C)
    so = cross_val_predict(meta, Xo, y, cv=folds, method="predict_proba")
    meta.fit(Xo, y)
    return so, meta.predict_proba(Xt), macro_f1(y, so.argmax(1))

# Faithful sklearn API (slide 28: 'sklearn: StackingClassifier') over fast tree learners, internal cv.
api_stack = StackingClassifier(
    estimators=[("rf",  make_pipeline(SimpleImputer(strategy="median"),
                                      RandomForestClassifier(150, random_state=SEED, n_jobs=-1))),
                ("et",  make_pipeline(SimpleImputer(strategy="median"),
                                      ExtraTreesClassifier(150, random_state=SEED, n_jobs=-1))),
                ("hgb", HistGradientBoostingClassifier(max_iter=250, random_state=SEED))],
    final_estimator=LogisticRegression(max_iter=2000), cv=3, stack_method="predict_proba", n_jobs=-1)
api_oof = cross_val_predict(api_stack, Xtr_np, y, cv=folds, method="predict")
print(f"sklearn StackingClassifier (RF+ET+HistGB, LogReg blender) OOF = {macro_f1(y, api_oof):.5f}")
print("  -> tree-only stacking stays at the ~0.82 wall: a blender cannot create signal no member has.")

# DIAGNOSTIC (illustrative, NOT selection): layered stacks show WHERE the lift comes from -- trees-only
# stays at the wall; adding the strong cross-family members lifts it; adding decorrelated generatives
# lifts it again. We do NOT choose the deployed set by ranking these OOF numbers (that would be in-sample
# selection); the deployed set is fixed a-priori below and validated out-of-fold by the paired CV.
layers = {
    "L1 textbook trees [rf,et,gb,histgb,adaboost]":       ["rf", "et", "gb", "histgb", "adaboost"],
    "L2 + cross-family strong [+cat,+svm]":               ["rf", "et", "gb", "histgb", "adaboost", "catboost_tuned", "svm"],
    "L3 strong only [cat,svm]":                           ["catboost_tuned", "svm"],
    "L4 strong + decorrelated generative [+qda,+gmm2]":   ["catboost_tuned", "svm", "qda", "gmm2"],
}
print("diagnostic layers (illustrative; deployed set is fixed a-priori, see below):")
for name, names in layers.items():
    print(f"  {name:52s} OOF {stack_oof_test(names)[2]:.5f}")

# DEPLOYED member set, fixed A-PRIORI from the step-11 eligibility analysis (NOT from the ranking above):
# the two STRONG cross-family learners (CatBoost, SVM-RBF = the sub06 pair) + the two MOST-DECORRELATED
# strong-enough generatives (QDA errcorr ~0.61/0.65, GMM2 ~0.52/0.56 vs CatBoost/SVM, per step 11).
# Nystroem is excluded a-priori (too correlated with SVM, ~0.80); LDA/GNB excluded (too weak).
STACK_MEMBERS = ["catboost_tuned", "svm", "qda", "gmm2"]
stack_oof, stack_test, stack_macro = stack_oof_test(STACK_MEMBERS)
print(f"\nDEPLOYED stack (a-priori set) {STACK_MEMBERS}")
print(f"  OOF macro-F1 = {stack_macro:.5f} | per-class = {per_class_f1(y, stack_oof.argmax(1))}")
print(f"  vs anchor {anchor:.5f} | vs sub06 0.83269 (step-10, frozen) | noise floor {NOISE:.5f}")
print("  C-stability on shared folds (in-sample sanity check, NOT selection):",
      {C: round(stack_oof_test(STACK_MEMBERS, C)[2], 5) for C in [0.1, 0.3, 1.0, 3.0]})

# One-lever WHAT-IF, shown but NOT shipped (MLP set aside by preference): adding the saved MLP.
m_mlp = stack_oof_test(STACK_MEMBERS + ["mlp"])[2]
print(f"\nwhat-if + MLP (set aside): {m_mlp:.5f} (+{m_mlp - stack_macro:.5f}) -> the largest remaining lever.")
np.save(ART / "stack_ch7_oof.npy", stack_oof); np.save(ART / "stack_ch7_test.npy", stack_test)
log_result("12_textbook_ensembles", "stacking_cat-svm-qda-gmm2", "features_v1", stack_macro,
           per_class_f1(y, stack_oof.argmax(1)),
           f"LogReg-meta stack; api(tree-only)={macro_f1(y, api_oof):.5f}; +mlp what-if={m_mlp:.5f}")

## (7) Honest paired CV — does the stack really help? (project discipline)
`RepeatedStratifiedKFold(5×3)`, every arm refit on identical fresh folds, leak-free. The stack is compared against the single-CatBoost anchor and the `sub06` Cat+SVM bar — read the paired delta, never the lucky seed-42 split.

In [ ]:
# Honest headline: paired RepeatedStratifiedKFold(5x3, seed=2026); all arms on IDENTICAL fresh folds,
# fully leak-free (CatBoost at fixed iterations; SVM/QDA/GMM scaler+impute inside the train fold; the
# blender fit out-of-fold per repeat). Bar = sub06 (Cat+SVM 0.5/0.5); the stack adds QDA+GMM via LogReg.
bp = json.load(open(ART / "catboost_best_params.json"))
SVM_CFG = json.load(open(ART / "svm_grid.json"))["topk"][0]      # best single SVM-RBF (C, gamma)
N_REP, CAT_ITERS = 3, 900
splits = list(RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REP, random_state=2026).split(np.zeros(len(y)), y))

def make_cat_fixed(seed=SEED):
    return CatBoostClassifier(loss_function="MultiClass", eval_metric="TotalF1:average=Macro",
        iterations=CAT_ITERS, learning_rate=bp["lr"], depth=bp["depth"], l2_leaf_reg=bp["l2"],
        random_seed=seed, allow_writing_files=False, thread_count=-1, verbose=False)

def fam_oof(make_est, the_folds, scale):
    oo = np.zeros((len(y), len(CLASSES)))
    for tr, va in the_folds:
        Xt, Xv = Xtr_np[tr].copy(), Xtr_np[va].copy()
        med = np.nanmedian(Xt[:, EOG_I])
        for M in (Xt, Xv):
            c = M[:, EOG_I]; c[np.isnan(c)] = med
        if scale:
            sc = StandardScaler().fit(Xt); Xt, Xv = sc.transform(Xt), sc.transform(Xv)
        m = make_est(); m.fit(Xt, y[tr]); oo[va] = aligned(m, Xv)
    return oo

cat_s, cs_s, st_s = [], [], []
for r in range(N_REP):
    fs = splits[r * N_FOLDS:(r + 1) * N_FOLDS]
    cat_o = np.zeros((len(y), len(CLASSES)))
    for tr, va in fs:
        cb = make_cat_fixed(); cb.fit(Xtr.iloc[tr], y[tr]); cat_o[va] = aligned(cb, Xtr.iloc[va])
    svm_o = fam_oof(lambda: SVC(C=SVM_CFG["C"], gamma=SVM_CFG["gamma"], kernel="rbf",
                                probability=True, random_state=SEED), fs, scale=True)
    qda_o = fam_oof(lambda: QuadraticDiscriminantAnalysis(reg_param=0.1), fs, scale=True)
    gmm_o = fam_oof(lambda: GMMClassifier(2), fs, scale=True)
    Xo = np.hstack([cat_o, svm_o, qda_o, gmm_o])
    st_o = cross_val_predict(LogisticRegression(max_iter=4000, C=1.0), Xo, y, cv=fs, method="predict_proba")
    cat_s.append(macro_f1(y, cat_o.argmax(1)))
    cs_s.append(macro_f1(y, (0.5 * cat_o + 0.5 * svm_o).argmax(1)))
    st_s.append(macro_f1(y, st_o.argmax(1)))
    print(f"  repeat {r + 1}: Cat {cat_s[-1]:.5f} | Cat+SVM {cs_s[-1]:.5f} | STACK {st_s[-1]:.5f}")

cat_s, cs_s, st_s = map(np.array, (cat_s, cs_s, st_s))
d_cat, d_sub06 = st_s - cat_s, st_s - cs_s
print(f"\nPAIRED {N_REP}x{N_FOLDS} (seed 2026):")
print(f"  single Cat : {cat_s.mean():.5f} +/- {cat_s.std():.5f}")
print(f"  Cat+SVM    : {cs_s.mean():.5f} +/- {cs_s.std():.5f}")
print(f"  STACK      : {st_s.mean():.5f} +/- {st_s.std():.5f}")
print(f"  delta STACK vs Cat   : {d_cat.mean():+.5f}  (positive in {int((d_cat > 0).sum())}/{N_REP})")
print(f"  delta STACK vs sub06 : {d_sub06.mean():+.5f}  (positive in {int((d_sub06 > 0).sum())}/{N_REP})")
print(f"  STACK beats single Cat by > noise floor ({NOISE:.5f})? {bool(d_cat.mean() > NOISE)}")
json.dump({"n_repeats": N_REP, "cat_mean": round(float(cat_s.mean()), 5),
           "catsvm_mean": round(float(cs_s.mean()), 5), "stack_mean": round(float(st_s.mean()), 5),
           "stack_std": round(float(st_s.std()), 5), "delta_vs_cat": round(float(d_cat.mean()), 5),
           "delta_vs_sub06": round(float(d_sub06.mean()), 5), "pos_vs_cat": int((d_cat > 0).sum()),
           "pos_vs_sub06": int((d_sub06 > 0).sum())}, open(ART / "stack_ch7_paired_cv.json", "w"), indent=2)
log_result("12_textbook_ensembles", "stacking_pairedCV", "features_v1", float(st_s.mean()),
           per_class_f1(y, stack_oof.argmax(1)),
           f"paired {N_REP}x{N_FOLDS}: STACK {st_s.mean():.5f} vs Cat {cat_s.mean():.5f} (delta {d_cat.mean():+.5f})")

## Submission `sub08` — Géron-ch7 stacking ensemble
Shipped as the requested principled variant. **Honest read of the paired CV:** the stack **ties** `sub06` within the ±0.0017 noise floor → *not* a score win, but a cleaner, textbook-faithful construction worth keeping as a **private-LB diversity hedge**. It clears the single-CatBoost anchor robustly and is far above the naive soft-voting `sub02` (0.81952). The set-aside MLP path (~0.837 OOF, +0.003) is the largest remaining lever — kept on disk, not shipped, by preference.

In [ ]:
# Submission sub08 -- the Geron-ch7 STACKING ensemble (the requested principled variant).
# HONEST STANCE (read the paired CV, not the lucky split): on the fresh paired RepeatedStratifiedKFold
# the stack ties sub06 WITHIN the noise floor -> NOT a score win. It is shipped as a DIFFERENT,
# textbook-faithful construction (a private-LB diversity hedge), it clears the single-CatBoost anchor
# robustly, and it is far above the naive soft-voting sub02 (0.81952). The set-aside MLP path reaches
# ~0.837 OOF (~+0.003, the largest remaining lever), kept on disk by preference and NOT shipped.
SUB06_OOF = json.load(open(ART / "svm_ensemble.json")).get("ensemble_macro", 0.83269)
print(f"sub08 stacking | OOF {stack_macro:.5f} vs sub06 {SUB06_OOF:.5f}")
print(f"  paired 5x3 vs sub06 : STACK {st_s.mean():.5f} vs {cs_s.mean():.5f}  (delta {d_sub06.mean():+.5f}, within noise {NOISE:.5f}) -> tie, not a win")
print(f"  paired 5x3 vs anchor: STACK {st_s.mean():.5f} vs Cat {cat_s.mean():.5f}  (delta {d_cat.mean():+.5f}, {int((d_cat > 0).sum())}/{N_REP} repeats) -> robust over single CatBoost")
name8 = "sub08_Geron_Ch7_StackingEnsemble.csv"
pred8 = stack_test.argmax(1).astype(int)
sub8 = pd.DataFrame({"id": test_ids, "sleep_stage": pred8})
assert sub8.shape == (5000, 2)
assert sub8["id"].tolist() == list(range(9000, 14000))
assert sub8["sleep_stage"].isin(CLASSES).all()
sub8.to_csv(SUB / name8, index=False)
print("WROTE", SUB / name8, "| OOF", round(float(stack_macro), 5))
print("class counts:", sub8["sleep_stage"].value_counts().sort_index().to_dict())
base = np.load(ART / "svm_ensemble_test.npy")               # sub06 (Cat+SVM) test probabilities
print("sub08 vs sub06 differ on", int((stack_test.argmax(1) != base.argmax(1)).sum()), "of 5000 rows")

### Takeaways
- **Stacking > voting**, exactly as the chapter argues — but a blender only helps when the base models are **diverse in strength and errors**. Tree-only stacking stays at the wall; the cross-family stack reaches the ceiling.
- Everything is honest: shared folds, out-of-fold blender, a-priori member set & meta `C`, paired CV as the headline, and a within-noise result reported as such (no overclaiming).
- The textbook methods, applied rigorously, **reproduce the project's whole arc** — from the soft-voting failure to the cross-family ceiling — in one self-contained Chapter-7 notebook.
- One lever remains (the set-aside MLP, ≈+0.003); the rest of the gap is the Deep↔Light/REM overlap in the synthetic data.